## Position Refinement (DLS, I14, test target)

In [ ]:
import h5py
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import ptypy.utils as u

In [ ]:
def color_wheel(N = 2056):
    sh = (N, int(N/1.))
    psize = (1./sh[0], 1./sh[1])    
    X, Y = np.indices(sh) * np.asarray(psize).reshape((len(sh),) + len(sh)*(1,))
    comcax = X * np.exp(2j*np.pi*Y)
    colors = np.array(u.imsave(comcax)) / 256.
    color_tuples = np.array([colors[:,:,0].flatten(), 
                             colors[:,:,1].flatten(), 
                             colors[:,:,2].flatten()]).transpose()
    S = np.zeros((N, N))
    r = np.linspace(0,1,N+1)
    t = np.linspace(0,2*np.pi,N+1)
    R,T = np.meshgrid(t,r)
    return R,T,S,color_tuples

## Loading reconstructions from .ptyr (HDF5) files

In [ ]:
path_to_recons_1 = "/dls/science/users/iat69393/ptypy-paper/output/i14_logo_posref_true_DM_cupy_0300.ptyr"
path_to_recons_2 = "/dls/science/users/iat69393/ptypy-paper/output/i14_logo_posref_false_DM_cupy_0300.ptyr"

In [ ]:
c = 60
with h5py.File(path_to_recons_1, "r") as f:
    obj1 = f["content/obj/Sscan_00G00/data"][0,c:-c,c:-c][::-1]
    obj1 *= np.exp(-1j*(1.5 - np.median(np.angle(obj1))))
    obj1 /= np.median(np.abs(obj1))
    prb1 = f["content/probe/Sscan_00G00/data"][0][::-1]
    pixelsize = f['content/obj/Sscan_00G00/_psize'][0]*1e9
with h5py.File(path_to_recons_2, "r") as f:
    obj2 = f["content/obj/Sscan_00G00/data"][0,c:-c,c:-c][::-1]
    obj2 *= np.exp(-1j*(1.5 - np.median(np.angle(obj2))))
    obj2 /= np.median(np.abs(obj2))
NY,NX = obj1.shape

## Figure comparing results with/without position refinement

In [ ]:
fig, axd = plt.subplot_mosaic([["A","A","B","B"],
                               ["A","A","B","B"],
                               ["E","F","G","H"]],
                               figsize=(12,6.5), constrained_layout=False, dpi=300,
                               gridspec_kw=dict(width_ratios=[1,1,1,1], height_ratios=[1,1,0.6]))
A = u.PtyAxis(axd["A"], channel="c", vmin=0.7, vmax=1.3)
A.set_data(obj2)
A.ax.axis("off")
A.ax.add_patch(plt.Rectangle((450,570),200,100, fill=0, lw=1, color="r"))
A.ax.add_patch(plt.Rectangle((450,900),300,150, fill=0, lw=1, color="b"))
A.ax.text(50,50, 'a', va='center', ha='center', color='w', fontsize=15, fontweight='bold')
A.ax.add_patch(plt.Rectangle((1550,50),5000/pixelsize,10, color='w'))
A.ax.text(1550+5000/pixelsize/2,80,r"%d $\mathrm{\mu}$m" %(5000/1e3), color='w', va='top', ha='center', fontsize=10)

B = u.PtyAxis(axd["B"], channel="c", vmin=0.7, vmax=1.3)
B.set_data(obj1)
B.ax.axis("off")
B.ax.add_patch(plt.Rectangle((450,570),200,100, fill=0, lw=2,ls=":", color="r"))
B.ax.add_patch(plt.Rectangle((450,900),300,150, fill=0, lw=2,ls=":", color="b"))
B.ax.text(50,50, 'b', va='center', ha='center', color='w', fontsize=15, fontweight='bold')

# C = u.PtyAxis(axd["C"], channel="c")
# C.set_data(prb1)
# C.ax.axis("off")
# C.ax.text(8,8, 'c', va='center', ha='center', color='w', fontsize=15, fontweight='bold')
# C.ax.add_patch(plt.Rectangle((85,5),1000/pixelsize,2, color='w'))
# C.ax.text(85+1000/pixelsize/2,10,r"%d $\mathrm{\mu}$m" %(1000/1e3), color='w', va='top', ha='center', fontsize=10)

# D = u.PtyAxis(axd["D"], channel="a", cmap="gray")
# D.set_data(np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(prb1))))
# D.ax.axis("off")
# D.ax.text(8,8, 'd', va='center', ha='center', color='w', fontsize=15, fontweight='bold')

E = u.PtyAxis(axd["E"], channel="c", vmin=0.7, vmax=1.3)
E.set_data(obj2[570:670,450:650])

F = u.PtyAxis(axd["F"], channel="c", vmin=0.7, vmax=1.3)
F.set_data(obj2[900:1050,450:750])

G = u.PtyAxis(axd["G"], channel="c", vmin=0.7, vmax=1.3)
G.set_data(obj1[570:670,450:650])

H = u.PtyAxis(axd["H"], channel="c", vmin=0.7, vmax=1.3)
H.set_data(obj1[900:1050,450:750])

for a,c,s in zip(["E","F","G","H"],["r","b","r","b"],["-","-",":",":"]):
    axd[a].set_xticks([])
    axd[a].set_yticks([])
    for x in ["bottom","top","right","left"]:
        axd[a].spines[x].set_color(c)
        axd[a].spines[x].set_linestyle(s)
        axd[a].spines[x].set_linewidth(2)

#axd["I"].axis("off")
caxm  = fig.add_axes([0.92,0.15,0.1,0.1], polar=True)
R,T,S,CC = color_wheel(N=1024)
caxm.pcolormesh(R,T,S,color=CC)
caxm.set_thetagrids([0,90,180,270], labels=[r'$0$', '$\mathrm{\pi/2}$','$\mathrm{\pi}$','$\mathrm{3\pi/4}$'])
caxm.spines['polar'].set_visible(False)
caxm.set_yticks([])
caxm.tick_params('x', labelsize=10, pad=-1)
plt.savefig("/dls/science/users/iat69393/ptypy-paper/figures/fig_dls_i14_test_target.png", dpi=300, bbox_inches="tight")
plt.show()

## Performance metrics

In [ ]:
import json
nframes = 200000
npixel = 128
npixel2 = npixel**2
niterations = 300
with open("./summary_dls_i14_test_target.json", "r") as f:
    metrics = json.load(f)

host = metrics["host"]
total = sum([v for v in metrics["benchmark"].values()])
time_prep_per_pixel = metrics["benchmark"]["data_load"] / nframes / npixel2 * 1e9
time_iterate_per_pixel = metrics["benchmark"]["engine_iterate"] / nframes / npixel2 / niterations * 1e9
print(f"Total: {total:.0f} s | Preparation: {time_prep_per_pixel:.2f} ns/px | Iterate: {time_iterate_per_pixel:.2f} ns/px")
print(f"Cluster node: {host}")